In [11]:
from pynwb import NWBHDF5IO

path = r"C:\Users\aksel\.ethograph\example_data\Moll2025\.ethograph\alignment.nwb"

io = NWBHDF5IO(path, mode="r", load_namespaces=True)
nwb = io.read()

print(nwb.trials)
print(nwb.trials.colnames)

# Access the actual trial table
df = nwb.trials.to_dataframe()
print(df)

trials pynwb.epoch.TimeIntervals at 0x2043919406656
Fields:
  colnames: ['start_time' 'stop_time' 'trial' 'video_cam-1' 'pose_cam-1']
  columns: (
    start_time <class 'hdmf.common.table.VectorData'>,
    stop_time <class 'hdmf.common.table.VectorData'>,
    trial <class 'hdmf.common.table.VectorData'>,
    video_cam-1 <class 'hdmf.common.table.VectorData'>,
    pose_cam-1 <class 'hdmf.common.table.VectorData'>
  )
  description: experimental trials
  id: id <class 'hdmf.common.table.ElementIdentifiers'>

('start_time', 'stop_time', 'trial', 'video_cam-1', 'pose_cam-1')
    start_time  stop_time  trial                     video_cam-1  \
id                                                                 
0          0.0      5.845     41  2024-12-17_115_Crow1-cam-1.mp4   
1          0.0      5.385    115  2024-12-18_041_Crow1-cam-1.mp4   

                           pose_cam-1  
id                                     
0   2024-12-17_115_Crow1-cam-1DLC.csv  
1   2024-12-18_041_Crow1-cam-

In [5]:
import ethograph as eto

path = r"C:\Users\aksel\.ethograph\example_data\Moll2025\2024-12-18_041_Crow1-cam-1.mp4"

import cv2

cap = cv2.VideoCapture(path)
num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(num_frames)

path2 = r"C:\Users\aksel\.ethograph\example_data\Moll2025\Trial_data.nc"
dt = eto.open(path2)
dt.itrial(0)

1077


<xarray.DatasetView> Size: 597kB
Dimensions:               (time: 1077, space: 3, keypoint: 3, individual: 1,
                           s3d_dims: 20, RGB: 3)
Coordinates:
  * time                  (time) float64 9kB 0.0 0.005 0.01 ... 5.37 5.375 5.38
  * space                 (space) <U1 12B 'x' 'y' 'z'
  * keypoint              (keypoint) <U8 96B 'beakTip' 'stickTip' 'pellet'
  * individual            (individual) <U5 20B 'Crow1'
Dimensions without coordinates: s3d_dims, RGB
Data variables: (12/14)
    position              (time, space, keypoint, individual) float64 78kB ...
    confidence            (time, keypoint, individual) float32 13kB ...
    velocity              (time, space, keypoint, individual) float64 78kB ...
    speed                 (time, keypoint, individual) float64 26kB ...
    acceleration          (time, space, keypoint, individual) float64 78kB ...
    pellet_beakTip_dist   (time) float64 9kB ...
    ...                    ...
    disp_stickTip_dist    (time, individual) float64 9kB ...
    s3d                   (time, s3d_dims) float64 172kB ...
    speed_troughs         (keypoint, individual, time) int8 3kB ...
    speed_turning_points  (keypoint, individual, time) int8 3kB ...
    angles                (keypoint, individual, time) float64 26kB ...
    angle_rgb             (keypoint, individual, time, RGB) float64 78kB ...
Attributes:
    source_software:  DeepLabCut
    ds_type:          poses
    fps:              200.0
    time_unit:        seconds
    source_file:      c:/Users/aksel/Documents/Code/ethograph/data/Moll2025/2...
    trial:            41
    pellet_position:  right

In [14]:
path = r"C:\Users\aksel\Documents\AI_data\derivatives\sub-01_id-Ivy\ses-000_date-20260421_01\behav\.ethograph\alignment.nwb"

from pynwb import NWBHDF5IO

nwb = NWBHDF5IO(path, 'r').read()


In [15]:
from __future__ import annotations

from pathlib import Path

import cv2
import pandas as pd
from movement.io import load_poses


def count_mp4_frames(video_path: Path) -> int:
    capture = cv2.VideoCapture(str(video_path))
    try:
        return int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    finally:
        capture.release()


def count_pose_frames(pose_path: Path) -> int:
    dataset = load_poses.from_dlc_file(pose_path)
    return int(dataset.sizes["time"])


def find_dlc_file(video_path: Path, dlc_dir: Path) -> Path:
    matches = sorted(dlc_dir.glob(f"{video_path.stem}DLC*.h5"))
    if not matches:
        raise FileNotFoundError(f"No DLC file found for {video_path.name} in {dlc_dir}")
    return matches[0]


def add_frame_counts(
    df: pd.DataFrame,
    video_dir: Path,
    dlc_dir: Path,
    video_columns: tuple[str, ...] = ("video_cam-1", "video_cam-2"),
) -> pd.DataFrame:
    result = df.copy()
    for column in video_columns:
        video_paths = result[column].apply(lambda name: video_dir / name)
        result[f"{column}_mp4_frames"] = video_paths.apply(count_mp4_frames)
        dlc_paths = video_paths.apply(lambda p: find_dlc_file(p, dlc_dir))
        result[f"{column}_dlc_frames"] = dlc_paths.apply(count_pose_frames)
        result[f"{column}_frame_diff"] = (
            result[f"{column}_mp4_frames"] - result[f"{column}_dlc_frames"]
        )
    return result


if __name__ == "__main__":
    video_dir = Path(r"C:\Users\aksel\Documents\VidData\20260421_01_Ivy")
    dlc_dir = Path(r"E:\Backup\Akseli\AI_data\rawdata\sub-01_id-Ivy\ses-000_date-20260421_01\behav\dlc")

    df = nwb.trials.to_dataframe()
    df = add_frame_counts(df, video_dir, dlc_dir)
    print(df)

     start_time    stop_time  trial                   video_cam-1  \
id                                                                  
0     43.086400    73.131400      1  2026-04-21_001_Ivy-cam-1.mp4   
1    105.831567   111.576567      2  2026-04-21_002_Ivy-cam-1.mp4   
2    201.798700   214.783700      5  2026-04-21_005_Ivy-cam-1.mp4   
3    286.387367   298.277367      6  2026-04-21_006_Ivy-cam-1.mp4   
4    304.527433   316.347433      7  2026-04-21_007_Ivy-cam-1.mp4   
..          ...          ...    ...                           ...   
89  4432.139767  4446.239767    103  2026-04-21_103_Ivy-cam-1.mp4   
90  4450.694833  4463.239833    104  2026-04-21_104_Ivy-cam-1.mp4   
91  4469.489800  4482.124800    105  2026-04-21_105_Ivy-cam-1.mp4   
92  4488.359733  4502.039733    106  2026-04-21_106_Ivy-cam-1.mp4   
93  4508.294133  4521.259133    107  2026-04-21_107_Ivy-cam-1.mp4   

                     video_cam-2  \
id                                 
0   2026-04-21_001_Ivy-cam-2.m

In [ ]:
from pathlib import Path

frame_count = count_mp4_frames(Path(path))
print(frame_count)


,start_time,stop_time,trial,video_cam-1,video_cam-2,pose_cam-1,pose_cam-2,video_cam-1_mp4_frames,video_cam-1_dlc_frames,video_cam-1_frame_diff,video_cam-2_mp4_frames,video_cam-2_dlc_frames,video_cam-2_frame_diff
id,,,,,,,,,,,,,
0,43.086400,73.131400,1,2026-04-21_001_Ivy-cam-1.mp4,2026-04-21_001_Ivy-cam-2.mp4,2026-04-21_001_Ivy-cam-1DLC_resnet50_Felix_cro...,2026-04-21_001_Ivy-cam-2DLC_resnet50_Felix_cro...,4598,6010,-1412,4598,6010,-1412
1,105.831567,111.576567,2,2026-04-21_002_Ivy-cam-1.mp4,2026-04-21_002_Ivy-cam-2.mp4,2026-04-21_002_Ivy-cam-1DLC_resnet50_Felix_cro...,2026-04-21_002_Ivy-cam-2DLC_resnet50_Felix_cro...,1150,1150,0,1150,1150,0
2,201.798700,214.783700,5,2026-04-21_005_Ivy-cam-1.mp4,2026-04-21_005_Ivy-cam-2.mp4,2026-04-21_005_Ivy-cam-1DLC_resnet50_Felix_cro...,2026-04-21_005_Ivy-cam-2DLC_resnet50_Felix_cro...,2598,2598,0,2598,2598,0
3,286.387367,298.277367,6,2026-04-21_006_Ivy-cam-1.mp4,2026-04-21_006_Ivy-cam-2.mp4,2026-04-21_006_Ivy-cam-1DLC_resnet50_Felix_cro...,2026-04-21_006_Ivy-cam-2DLC_resnet50_Felix_cro...,967,2379,-1412,967,2379,-1412
4,304.527433,316.347433,7,2026-04-21_007_Ivy-cam-1.mp4,2026-04-21_007_Ivy-cam-2.mp4,2026-04-21_007_Ivy-cam-1DLC_resnet50_Felix_cro...,2026-04-21_007_Ivy-cam-2DLC_resnet50_Felix_cro...,953,2365,-1412,953,2365,-1412
...,...,...,...,...,...,...,...,...,...,...,...,...,...
89,4432.139767,4446.239767,103,2026-04-21_103_Ivy-cam-1.mp4,2026-04-21_103_Ivy-cam-2.mp4,2026-04-21_103_Ivy-cam-1DLC_resnet50_Felix_cro...,2026-04-21_103_Ivy-cam-2DLC_resnet50_Felix_cro...,1409,2821,-1412,1409,2821,-1412
90,4450.694833,4463.239833,104,2026-04-21_104_Ivy-cam-1.mp4,2026-04-21_104_Ivy-cam-2.mp4,2026-04-21_104_Ivy-cam-1DLC_resnet50_Felix_cro...,2026-04-21_104_Ivy-cam-2DLC_resnet50_Felix_cro...,1098,2510,-1412,1098,2510,-1412
91,4469.489800,4482.124800,105,2026-04-21_105_Ivy-cam-1.mp4,2026-04-21_105_Ivy-cam-2.mp4,2026-04-21_105_Ivy-cam-1DLC_resnet50_Felix_cro...,2026-04-21_105_Ivy-cam-2DLC_resnet50_Felix_cro...,1116,2528,-1412,1116,2528,-1412


In [7]:
print(df)

     start_time    stop_time  trial                   video_cam-1  \
id                                                                  
0     43.086400    73.131400      1  2026-04-21_001_Ivy-cam-1.mp4   
1    105.831567   111.576567      2  2026-04-21_002_Ivy-cam-1.mp4   
2    201.798700   214.783700      5  2026-04-21_005_Ivy-cam-1.mp4   
3    286.387367   298.277367      6  2026-04-21_006_Ivy-cam-1.mp4   
4    304.527433   316.347433      7  2026-04-21_007_Ivy-cam-1.mp4   
..          ...          ...    ...                           ...   
89  4432.139767  4446.239767    103  2026-04-21_103_Ivy-cam-1.mp4   
90  4450.694833  4463.239833    104  2026-04-21_104_Ivy-cam-1.mp4   
91  4469.489800  4482.124800    105  2026-04-21_105_Ivy-cam-1.mp4   
92  4488.359733  4502.039733    106  2026-04-21_106_Ivy-cam-1.mp4   
93  4508.294133  4521.259133    107  2026-04-21_107_Ivy-cam-1.mp4   

                     video_cam-2  \
id                                 
0   2026-04-21_001_Ivy-cam-2.m

In [6]:
df = nwb.trials.to_dataframe()
df

,start_time,stop_time,trial,video_cam-1,video_cam-2,pose_cam-1,pose_cam-2
id,,,,,,,
0,43.086400,73.131400,1,2026-04-21_001_Ivy-cam-1.mp4,2026-04-21_001_Ivy-cam-2.mp4,2026-04-21_001_Ivy-cam-1DLC_resnet50_Felix_cro...,2026-04-21_001_Ivy-cam-2DLC_resnet50_Felix_cro...
1,105.831567,111.576567,2,2026-04-21_002_Ivy-cam-1.mp4,2026-04-21_002_Ivy-cam-2.mp4,2026-04-21_002_Ivy-cam-1DLC_resnet50_Felix_cro...,2026-04-21_002_Ivy-cam-2DLC_resnet50_Felix_cro...
2,201.798700,214.783700,5,2026-04-21_005_Ivy-cam-1.mp4,2026-04-21_005_Ivy-cam-2.mp4,2026-04-21_005_Ivy-cam-1DLC_resnet50_Felix_cro...,2026-04-21_005_Ivy-cam-2DLC_resnet50_Felix_cro...
3,286.387367,298.277367,6,2026-04-21_006_Ivy-cam-1.mp4,2026-04-21_006_Ivy-cam-2.mp4,2026-04-21_006_Ivy-cam-1DLC_resnet50_Felix_cro...,2026-04-21_006_Ivy-cam-2DLC_resnet50_Felix_cro...
4,304.527433,316.347433,7,2026-04-21_007_Ivy-cam-1.mp4,2026-04-21_007_Ivy-cam-2.mp4,2026-04-21_007_Ivy-cam-1DLC_resnet50_Felix_cro...,2026-04-21_007_Ivy-cam-2DLC_resnet50_Felix_cro...
...,...,...,...,...,...,...,...
89,4432.139767,4446.239767,103,2026-04-21_103_Ivy-cam-1.mp4,2026-04-21_103_Ivy-cam-2.mp4,2026-04-21_103_Ivy-cam-1DLC_resnet50_Felix_cro...,2026-04-21_103_Ivy-cam-2DLC_resnet50_Felix_cro...
90,4450.694833,4463.239833,104,2026-04-21_104_Ivy-cam-1.mp4,2026-04-21_104_Ivy-cam-2.mp4,2026-04-21_104_Ivy-cam-1DLC_resnet50_Felix_cro...,2026-04-21_104_Ivy-cam-2DLC_resnet50_Felix_cro...
91,4469.489800,4482.124800,105,2026-04-21_105_Ivy-cam-1.mp4,2026-04-21_105_Ivy-cam-2.mp4,2026-04-21_105_Ivy-cam-1DLC_resnet50_Felix_cro...,2026-04-21_105_Ivy-cam-2DLC_resnet50_Felix_cro...
